# Merge the v4-mixed-r16 adapter at λ=0.75 and publish `winhsss/Reworkwhisper-large-v5`

Produces a standalone Whisper checkpoint at `winhsss/Reworkwhisper-large-v5` --
loadable with `WhisperForConditionalGeneration.from_pretrained`, no `peft`, no
`adapter_config.json`.

**The repo is deleted and recreated under the same name, by hand, before this
notebook runs.** That is the decision this notebook is written around: an empty
repo cannot leave a stale λ=0.25 adapter file behind, so nothing here deletes
remote files. Cell 4 verifies the deletion actually happened and stops if it did
not. The λ=0.25 adapter survives locally in `Outputs/outputs_v4-mixed-r16.zip`;
the λ=0.75 one in `Outputs/lambda075-adapter.zip`. Neither is recoverable from
the Hub once the repo is gone.

λ=0.75 is a **human override** of `select_lambda`, which still picks 0.5 -- the
rule optimises val CER against OOD CER and cannot see `english_token_retention`,
the axis production regressed on. The gate at 0.75 passed every tier
(`experiments/v4-mixed-r16-lambda0.75/metrics/gate_results.json`,
`overall_pass: true`), and the model card records the override rather than
presenting 0.75 as the rule's pick. SESSIONS.md has the full λ curve.

**No accelerator needed.** The merge runs fp32 on CPU (highest precision; a
published checkpoint is written once and loaded many times). Set the notebook
accelerator to *None* and save the GPU quota.

**Before running:**
- Delete `winhsss/Reworkwhisper-large-v5` on the Hub. Do **not** recreate it by
  hand -- `push_adapter` calls `create_repo(private=True, exist_ok=True)`, and
  `exist_ok=True` cannot lower the visibility of a repo somebody already made
  public. Letting the script create it is the only path that guarantees private.
- Add `HF_TOKEN` (write scope on `winhsss/`) under Add-ons -> Secrets.
- Attach the Kaggle Dataset holding the `v4-mixed-r16` run (the one with
  `outputs/v4-mixed-r16/adapter/`). Its adapter is baked at λ=0.25, but λ lives
  only in `adapter_config.json`'s `lora_alpha` -- the weights are byte-identical
  at every λ (verified: sha256 `db5f92da...` matches `Outputs/lambda075-adapter.zip`).
  Cell 3b checks that hash and re-bakes λ=0.75 locally, so no 110 MB upload is
  needed. If the hash ever fails, upload `Outputs/lambda075-adapter.zip` as a
  dataset and point `SRC_ADAPTER` at it instead.
- Push `main` first -- Cell 1 clones GitHub, not your local tree. It needs
  `scripts/merge_and_push.py`, `experiments/v4-mixed-r16-lambda0.75/` and
  `experiments/v3-r16-lambda0.5/`.

**Disk:** ~6.2 GB base download into `~/.cache/huggingface` plus ~6.2 GB for the
merged fp32 output under `/kaggle/working`.

**Reusing the name does not invalidate anyone's cache.** A client that already
downloaded the old `winhsss/Reworkwhisper-large-v5` keeps serving λ=0.25 from
`~/.cache/huggingface/hub/models--winhsss--Reworkwhisper-large-v5` until that
directory is removed or `force_download=True` is passed. Every consumer has to
be told; the repo name alone will not tell them.

## 1. Clone / update the repo

In [ ]:
import os

REPO_URL = "https://github.com/egoist-minh/Reworkwhisper-finetune.git"
REPO_DIR = "/kaggle/working/Reworkwhisper-finetune"

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull origin main
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

In [ ]:
!pip install -q -r requirements.txt

## 2. Token

Read from this notebook's Secrets, never hardcoded and never written into any
artifact. Needs **write** scope on `winhsss/`.

In [ ]:
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["TRANSFORMERS_AUTO_CONVERSION"] = "0"
print("HF_TOKEN set")

## 3. Locate the run adapter

Mount nesting differs per dataset -- read the path off this cell, never from a
previous notebook. Looking for the directory that holds
`adapter_config.json` + `adapter_model.safetensors`.

In [ ]:
!ls -la /kaggle/input/datasets/*/*

### 3b. Verify the weights, then bake λ=0.75

The attached adapter is baked at **λ=0.25** (`lora_alpha` 8.0). That is not a
different adapter: `src/lora.py` ships λ as `lora_alpha <- λ * alpha` and leaves
`lora_B` bit-identical to training, so every λ of this run is the same
115,487,384 bytes of weights with one number changed in a JSON file.

So this cell does the two things that make reusing it safe:

1. **sha256 the weights** against the adapter the gate actually measured
   (`Outputs/lambda075-adapter.zip`). This is the whole safety argument -- if the
   hash matches, the directory written below is indistinguishable from the gated
   one. If it does not, this is some other run's adapter and nothing downstream
   would notice.
2. **Rewrite `lora_alpha` to λ x alpha**, reading `alpha` from the run's own
   `config.json` rather than hardcoding 32.

`check_provenance` still re-derives λ from the file this cell writes and
cross-checks it against `lambda_sweep.csv` and the gate's tier-2 CER. Nothing
here is trusted on the strength of this cell alone.

In [ ]:
import hashlib
import json
import shutil
from pathlib import Path

SRC_ADAPTER = "/kaggle/input/datasets/<user>/v5/v5/outputs/v4-mixed-r16/adapter"  # edit
RUN_DIR = "experiments/v4-mixed-r16-lambda0.75"
ADAPTER = "/kaggle/working/adapter-lambda0.75"
LAM = 0.75

# sha256 of adapter_model.safetensors in Outputs/lambda075-adapter.zip, which is
# byte-identical to the one in Outputs/outputs_v4-mixed-r16.zip -- lambda never
# touches the weights.
TRAINED_SHA = "db5f92dafacdbd0e2964aac00d1070a5373b60c70cf937cce9c83eff414c8dc6"

weights = Path(SRC_ADAPTER) / "adapter_model.safetensors"
digest = hashlib.sha256(weights.read_bytes()).hexdigest()
print(f"{weights.stat().st_size} bytes, sha256 {digest}")
assert digest == TRAINED_SHA, "not the v4-mixed-r16 weights the gate measured -- wrong dataset"

shutil.rmtree(ADAPTER, ignore_errors=True)
shutil.copytree(SRC_ADAPTER, ADAPTER)

alpha = json.loads(Path(f"{RUN_DIR}/config.json").read_text(encoding="utf-8"))["lora"]["alpha"]
cfg_path = Path(ADAPTER) / "adapter_config.json"
acfg = json.loads(cfg_path.read_text(encoding="utf-8"))
print(f"in:  lora_alpha={acfg['lora_alpha']} -> lambda {acfg['lora_alpha'] / alpha}")
acfg["lora_alpha"] = LAM * alpha
cfg_path.write_text(json.dumps(acfg, indent=2), encoding="utf-8")
print(f"out: lora_alpha={acfg['lora_alpha']} -> lambda {acfg['lora_alpha'] / alpha}")

## 4. Confirm the target repo is gone

The whole point of deleting it is that nothing from the λ=0.25 adapter can
survive. Verify that here, before spending an fp32 merge on it.

- `cannot list ... RepositoryNotFoundError` (or `HfHubHTTPError` / `404`) --
  deleted. Proceed.
- A file listing -- the deletion did not happen, or the repo was recreated by
  hand. Stop: recreating by hand also means `create_repo(exist_ok=True)` cannot
  make it private, and any listed `adapter_*` file would still load at λ=0.25.

In [ ]:
from huggingface_hub import HfApi

REPO_ID = "winhsss/Reworkwhisper-large-v5"

try:
    files = sorted(HfApi().list_repo_files(REPO_ID))
    print("\n".join(files))
    raise SystemExit(f"{REPO_ID} still exists with {len(files)} files -- delete it first")
except SystemExit:
    raise
except Exception as e:
    print(f"cannot list {REPO_ID}: {type(e).__name__}: {e}")
    print("-> repo is absent; merge_and_push will create it private")

## 5. Merge (and push once you have read the output)

`CONFIRM_PUSH = False` runs everything except the upload: it verifies provenance
(gate passed, adapter matches this run's base model / rank, baked λ is the sweep
row the gate scored), runs the production check, merges, checks the merged
logits against the unmerged PeftModel, writes the model card, and stops. Read
the printed card and the `max logit diff` line, then flip the flag and re-run --
the re-run redoes the merge from scratch (~5 min), it does not reuse the saved
folder.

Three lines to read before flipping:

- `provenance ok: ... lambda=0.75 tier2_ood_cer=0.033427`
- `production check: candidate cer=0.0422 vs production cer=0.0805 on 654 shared
  segments` -- 654 is the whole mixed test split; a smaller number means the join
  lost rows.
- `merged <n> LoRA layers; max logit diff vs unmerged ...` -- the diff must be
  below `--tol` (1e-3), and `<n>` must not be 0 (the script raises if it is).

No `--delete-remote-adapter`: Cell 4 established the repo is empty, so there is
nothing to delete. If you ever run this against a repo that still holds an
adapter, add the flag back -- otherwise `peft` callers keep loading the old λ.

In [ ]:
import subprocess

CONFIRM_PUSH = False   # flip to True after reading this cell's output yourself

PRODUCTION = "experiments/v3-r16-lambda0.5/predictions_tier1_in_domain.csv"
OUT_DIR = "/kaggle/working/merged-v4-mixed-r16-lambda0.75"

cmd = [
    "python", "-m", "scripts.merge_and_push",
    "--run-dir", RUN_DIR,
    "--adapter", ADAPTER,
    "--repo-id", REPO_ID,
    "--out", OUT_DIR,
    "--production-predictions", PRODUCTION,
]
if CONFIRM_PUSH:
    cmd += ["--confirm"]

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
if proc.wait() != 0:
    raise SystemExit(f"merge_and_push failed with exit code {proc.returncode}")

print(open(f"{OUT_DIR}/README.md", encoding="utf-8").read())

## 6. Verify the published repo loads standalone

Downloads the repo fresh (no adapter, no `peft` import) and decodes one silent
frame. This proves the recreated repo is self-sufficient; it says nothing about
CER -- those numbers come from the gate in the model card.

`force_download=True` matters here specifically because the repo name was
reused: without it this cell can pass by reading a cached copy of the *deleted*
λ=0.25 repo and prove nothing about what was just uploaded.

In [ ]:
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor

model = WhisperForConditionalGeneration.from_pretrained(
    REPO_ID, torch_dtype=torch.float16, force_download=True)
processor = WhisperProcessor.from_pretrained(REPO_ID, force_download=True)
print(sum(p.numel() for p in model.parameters()), "params,", next(model.parameters()).dtype)

silence = torch.zeros(16000, dtype=torch.float32).numpy()
feats = processor(silence, sampling_rate=16000, return_tensors="pt").input_features.half()
ids = model.generate(feats, language="vi", task="transcribe", max_new_tokens=16)
print(repr(processor.batch_decode(ids, skip_special_tokens=True)[0]))